In [1]:

"""
CN7 09 - Causal Process Change Features
=======================================

목적
----
지금까지:
- Group CV에서는 PR-AUC 약 0.58
- Purged Time-Block에서는 PR-AUC 약 0.04

즉 "센서 절대값"은 특정 생산구간에 과도하게 의존할 가능성이 있다.

이번 단계에서는:
    현재 공정값
    -
    직전 / 최근 정상적인 공정 수준(X only)

형태의 causal temporal feature를 만든다.

중요
----
1) 동일 sensor vector를 하나의 공정 cycle로 본다.
2) cycle 단위로 시간 정렬한다.
3) 모든 rolling baseline은 반드시 shift(1) 이후 계산한다.
4) Target / Reason / 미래 row는 절대 사용하지 않는다.
5) 장시간 공백이 있으면 새 session으로 초기화한다.
6) 모델 튜닝은 하지 않는다.
   Logistic + StandardScaler + class_weight="balanced" 고정.
7) 평가:
   - PR-AUC
   - Recall >= 80%에서 Precision
   - Recall >= 90%에서 Precision
   - Top 5% Precision / Recall
"""

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
)

warnings.filterwarnings("ignore")


# =============================================================================
# 0. 설정
# =============================================================================

CN7_WORK_DIR = Path(
    r"C:\Users\Administrator\Desktop\kamp\다시하는 cn7"
)

RAW_PATH = (
    CN7_WORK_DIR
    / "cn7_03_eda"
    / "cn7_train_raw_1211.csv"
)

OUT_DIR = (
    CN7_WORK_DIR
    / "cn7_09_causal_features"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TARGET = "PassOrFail"

SENSOR_COLS = [
    "Injection_Time",
    "Filling_Time",
    "Plasticizing_Time",
    "Cycle_Time",
    "Clamp_Close_Time",
    "Cushion_Position",
    "Plasticizing_Position",
    "Clamp_Open_Position",
    "Max_Injection_Speed",
    "Max_Screw_RPM",
    "Average_Screw_RPM",
    "Max_Injection_Pressure",
    "Max_Switch_Over_Pressure",
    "Max_Back_Pressure",
    "Average_Back_Pressure",
    "Barrel_Temperature_1",
    "Barrel_Temperature_2",
    "Barrel_Temperature_3",
    "Barrel_Temperature_4",
    "Barrel_Temperature_5",
    "Barrel_Temperature_6",
    "Hopper_Temperature",
    "Mold_Temperature_3",
    "Mold_Temperature_4",
]

# 생산 cycle이 약 1분 단위이므로,
# 10분 이상 공백이면 새로운 생산 session으로 간주
SESSION_GAP_MINUTES = 10

ROLL_WINDOWS = [3, 5, 10]

GROUP_SEEDS = [
    42,
    62,
    82,
]

N_SPLITS = 5

TIME_BLOCK_MINUTES = 10
PURGE_MINUTES = 10

LOGREG_C = 0.1


# =============================================================================
# 유틸
# =============================================================================

def banner(title):
    print(
        "\n"
        + "=" * 118
    )
    print(title)
    print(
        "=" * 118
    )


def make_model():
    return Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="median",
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "model",
            LogisticRegression(
                C=LOGREG_C,
                class_weight="balanced",
                max_iter=3000,
                solver="liblinear",
                random_state=42,
            ),
        ),
    ])


def evaluate(y_true, score):
    y = np.asarray(
        y_true
    ).astype(int)

    s = np.asarray(
        score
    ).astype(float)

    valid = np.isfinite(s)

    y = y[valid]
    s = s[valid]

    result = {
        "n": len(y),
        "defects": int(
            y.sum()
        ),
    }

    if (
        len(y) == 0
        or len(
            np.unique(y)
        ) < 2
    ):
        result.update({
            "PR_AUC": np.nan,
            "ROC_AUC_ref": np.nan,
        })
        return result

    result[
        "PR_AUC"
    ] = float(
        average_precision_score(
            y,
            s,
        )
    )

    result[
        "ROC_AUC_ref"
    ] = float(
        roc_auc_score(
            y,
            s,
        )
    )

    # ---------------------------------------------------------
    # Recall constraint 운영점
    # ---------------------------------------------------------
    thresholds = np.sort(
        np.unique(s)
    )[::-1]

    threshold_rows = []

    for threshold in thresholds:

        pred = (
            s >= threshold
        ).astype(int)

        tn, fp, fn, tp = (
            confusion_matrix(
                y,
                pred,
                labels=[0, 1],
            ).ravel()
        )

        precision = (
            tp / (tp + fp)
            if (
                tp + fp
            ) > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if (
                tp + fn
            ) > 0
            else 0.0
        )

        threshold_rows.append({
            "threshold":
                float(
                    threshold
                ),
            "precision":
                float(
                    precision
                ),
            "recall":
                float(
                    recall
                ),
            "TP":
                int(tp),
            "FP":
                int(fp),
            "FN":
                int(fn),
            "predicted_positive":
                int(
                    tp + fp
                ),
        })

    threshold_df = pd.DataFrame(
        threshold_rows
    )

    for target_recall in [
        0.80,
        0.90,
    ]:

        cand = threshold_df[
            threshold_df[
                "recall"
            ] >= target_recall
        ].copy()

        key = int(
            target_recall * 100
        )

        if len(cand) == 0:

            result[
                f"R{key}_precision"
            ] = np.nan

            result[
                f"R{key}_recall"
            ] = np.nan

            result[
                f"R{key}_threshold"
            ] = np.nan

            continue

        cand = cand.sort_values(
            [
                "precision",
                "recall",
                "predicted_positive",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )

        r = cand.iloc[0]

        result[
            f"R{key}_precision"
        ] = float(
            r[
                "precision"
            ]
        )

        result[
            f"R{key}_recall"
        ] = float(
            r[
                "recall"
            ]
        )

        result[
            f"R{key}_threshold"
        ] = float(
            r[
                "threshold"
            ]
        )

        result[
            f"R{key}_TP"
        ] = int(
            r["TP"]
        )

        result[
            f"R{key}_FP"
        ] = int(
            r["FP"]
        )

    # ---------------------------------------------------------
    # Top 5%
    # ---------------------------------------------------------
    k = max(
        1,
        int(
            np.ceil(
                len(y) * 0.05
            )
        ),
    )

    order = np.argsort(
        -s
    )

    top = order[:k]

    tp = int(
        y[top].sum()
    )

    fp = int(
        k - tp
    )

    result[
        "Top5_precision"
    ] = float(
        tp / k
    )

    result[
        "Top5_recall"
    ] = float(
        tp / y.sum()
        if y.sum() > 0
        else np.nan
    )

    result[
        "Top5_TP"
    ] = tp

    result[
        "Top5_FP"
    ] = fp

    return result


# =============================================================================
# 1. 데이터 로드
# =============================================================================

banner(
    "CN7 09 - Causal Process Change Features"
)

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"없음: {RAW_PATH}"
    )

df = pd.read_csv(
    RAW_PATH
)

df[TARGET] = pd.to_numeric(
    df[TARGET],
    errors="raise",
).astype(int)

if "TimeStamp_dt" in df.columns:

    df[
        "TimeStamp_dt"
    ] = pd.to_datetime(
        df[
            "TimeStamp_dt"
        ],
        errors="coerce",
    )

else:

    df[
        "TimeStamp_dt"
    ] = pd.to_datetime(
        df[
            "TimeStamp"
        ],
        errors="coerce",
    )

if (
    df[
        "TimeStamp_dt"
    ].isna().any()
):
    raise RuntimeError(
        "TimeStamp 파싱 실패"
    )

if "Side" not in df.columns:

    s = (
        df[
            "PART_NAME"
        ]
        .astype(str)
        .str.upper()
    )

    df[
        "Side"
    ] = np.where(
        s.str.contains(
            r"\bRH\b",
            regex=True,
            na=False,
        ),
        "RH",
        "LH",
    )

df[
    "Side_RH"
] = (
    df[
        "Side"
    ] == "RH"
).astype(int)

constant_cols = [
    c
    for c in SENSOR_COLS
    if (
        c in df.columns
        and df[c].nunique(
            dropna=False
        ) <= 1
    )
]

base_sensors = [
    c
    for c in SENSOR_COLS
    if (
        c in df.columns
        and c not in constant_cols
    )
]

print(
    f"rows={len(df)}, "
    f"defects={int(df[TARGET].sum())}, "
    f"base_sensors={len(base_sensors)}"
)

print(
    "constant removed:",
    constant_cols,
)


# =============================================================================
# 2. 동일 센서벡터 = cycle
# =============================================================================

banner(
    "1) Cycle 재구성"
)

df[
    "sensor_group_id"
] = (
    df.groupby(
        base_sensors,
        dropna=False,
        sort=False,
    )
    .ngroup()
)

cycle = (
    df.groupby(
        "sensor_group_id",
        as_index=False,
    )
    .agg(
        {
            **{
                c: "first"
                for c in base_sensors
            },
            "TimeStamp_dt": "min",
        }
    )
)

cycle = (
    cycle
    .sort_values(
        [
            "TimeStamp_dt",
            "sensor_group_id",
        ]
    )
    .reset_index(
        drop=True
    )
)

cycle[
    "cycle_order"
] = np.arange(
    len(cycle)
)

cycle[
    "gap_seconds"
] = (
    cycle[
        "TimeStamp_dt"
    ]
    .diff()
    .dt.total_seconds()
)

new_session = (
    cycle[
        "gap_seconds"
    ].isna()
    |
    (
        cycle[
            "gap_seconds"
        ]
        > SESSION_GAP_MINUTES * 60
    )
)

cycle[
    "session_id"
] = (
    new_session
    .cumsum()
    .astype(int)
)

print(
    "row count:",
    len(df)
)

print(
    "cycle count:",
    len(cycle)
)

print(
    "session count:",
    cycle[
        "session_id"
    ].nunique()
)

print()

print(
    cycle.groupby(
        "session_id"
    )
    .agg(
        cycles=(
            "sensor_group_id",
            "size",
        ),
        start=(
            "TimeStamp_dt",
            "min",
        ),
        end=(
            "TimeStamp_dt",
            "max",
        ),
    )
    .to_string()
)


# =============================================================================
# 3. Causal feature 생성
# =============================================================================

banner(
    "2) Causal feature 생성"
)

temporal_cols = []

for sensor in base_sensors:

    # 각 session마다 직전 cycle
    lag1 = (
        cycle.groupby(
            "session_id"
        )[sensor]
        .shift(1)
    )

    col = (
        f"CF_{sensor}"
        "_delta1"
    )

    cycle[col] = (
        cycle[sensor]
        - lag1
    )

    temporal_cols.append(
        col
    )

    # ---------------------------------------------------------
    # rolling은 반드시 shift(1) 이후
    # ---------------------------------------------------------
    shifted = (
        cycle.groupby(
            "session_id"
        )[sensor]
        .shift(1)
    )

    for window in ROLL_WINDOWS:

        rolling_mean = (
            shifted.groupby(
                cycle[
                    "session_id"
                ]
            )
            .rolling(
                window=window,
                min_periods=2,
            )
            .mean()
            .reset_index(
                level=0,
                drop=True,
            )
        )

        dev_col = (
            f"CF_{sensor}"
            f"_dev{window}"
        )

        cycle[
            dev_col
        ] = (
            cycle[sensor]
            - rolling_mean
        )

        temporal_cols.append(
            dev_col
        )

        # zscore는 5/10에서만
        if window in [
            5,
            10,
        ]:

            rolling_std = (
                shifted.groupby(
                    cycle[
                        "session_id"
                    ]
                )
                .rolling(
                    window=window,
                    min_periods=2,
                )
                .std(
                    ddof=0
                )
                .reset_index(
                    level=0,
                    drop=True,
                )
            )

            z_col = (
                f"CF_{sensor}"
                f"_z{window}"
            )

            denom = (
                rolling_std
                .where(
                    rolling_std.abs()
                    > 1e-9,
                    np.nan,
                )
            )

            cycle[
                z_col
            ] = (
                cycle[
                    dev_col
                ]
                / denom
            )

            cycle[
                z_col
            ] = (
                cycle[
                    z_col
                ]
                .replace(
                    [
                        np.inf,
                        -np.inf,
                    ],
                    np.nan,
                )
                .clip(
                    -10,
                    10,
                )
            )

            temporal_cols.append(
                z_col
            )

print(
    "temporal features:",
    len(
        temporal_cols
    )
)

print(
    "예시:",
    temporal_cols[:15],
)


# =============================================================================
# 4. Row-level로 merge
# =============================================================================

banner(
    "3) Row-level merge"
)

merge_cols = [
    "sensor_group_id",
    "cycle_order",
    "session_id",
    "gap_seconds",
] + temporal_cols

model_df = df.merge(
    cycle[
        merge_cols
    ],
    on="sensor_group_id",
    how="left",
    validate="many_to_one",
)

if len(
    model_df
) != len(df):
    raise RuntimeError(
        "merge 후 row 수 변화"
    )

print(
    "model_df:",
    model_df.shape
)


# =============================================================================
# 5. Feature sets
# =============================================================================

banner(
    "4) Feature sets"
)

delta_cols = [
    c
    for c in temporal_cols
    if c.endswith(
        "_delta1"
    )
]

dev_cols = [
    c
    for c in temporal_cols
    if (
        "_dev" in c
        or "_z" in c
    )
]

FEATURE_SETS = {
    # 현재 baseline
    "base_side":
        base_sensors
        + ["Side_RH"],

    # 직전 cycle 변화만
    "delta_side":
        delta_cols
        + ["Side_RH"],

    # 최근 3/5/10 cycle 대비 편차
    "rolling_deviation_side":
        dev_cols
        + ["Side_RH"],

    # 변화량 전체
    "temporal_side":
        temporal_cols
        + ["Side_RH"],

    # 절대값 + 변화량
    "base_temporal_side":
        base_sensors
        + temporal_cols
        + ["Side_RH"],

    # Side 의존성 확인용
    "temporal_no_side":
        temporal_cols,
}

for name, cols in FEATURE_SETS.items():
    print(
        f"{name:28s}: "
        f"{len(cols)} features"
    )


# =============================================================================
# 6. Group CV
# =============================================================================

banner(
    "5) Repeated Group CV"
)

y = model_df[
    TARGET
]

groups = model_df[
    "sensor_group_id"
]

group_rows = []
group_predictions = {}

for feature_name, cols in FEATURE_SETS.items():

    X = model_df[
        cols
    ].astype(float)

    pred_sum = np.zeros(
        len(model_df)
    )

    pred_count = np.zeros(
        len(model_df),
        dtype=int,
    )

    for seed in GROUP_SEEDS:

        cv = StratifiedGroupKFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=seed,
        )

        for fold, (
            tr_idx,
            va_idx,
        ) in enumerate(
            cv.split(
                X,
                y,
                groups,
            ),
            start=1,
        ):

            model = make_model()

            model.fit(
                X.iloc[
                    tr_idx
                ],
                y.iloc[
                    tr_idx
                ],
            )

            pred = (
                model.predict_proba(
                    X.iloc[
                        va_idx
                    ]
                )[:, 1]
            )

            pred_sum[
                va_idx
            ] += pred

            pred_count[
                va_idx
            ] += 1

    oof = np.divide(
        pred_sum,
        pred_count,
        out=np.full(
            len(model_df),
            np.nan,
        ),
        where=(
            pred_count > 0
        ),
    )

    metrics = evaluate(
        y,
        oof,
    )

    group_predictions[
        feature_name
    ] = oof

    group_rows.append({
        "feature_set":
            feature_name,
        "n_features":
            len(cols),
        **metrics,
    })

    print(
        f"{feature_name:28s} | "
        f"PR={metrics['PR_AUC']:.4f} | "
        f"R80 P={metrics['R80_precision']:.3f}, "
        f"R={metrics['R80_recall']:.3f} | "
        f"R90 P={metrics['R90_precision']:.3f}, "
        f"R={metrics['R90_recall']:.3f} | "
        f"Top5 P={metrics['Top5_precision']:.3f}, "
        f"R={metrics['Top5_recall']:.3f}"
    )

group_df = pd.DataFrame(
    group_rows
)

group_df.to_csv(
    OUT_DIR
    / "group_results.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 7. Purged Time-Block CV
# =============================================================================

banner(
    "6) Purged Time-Block CV"
)

model_df[
    "time_block"
] = (
    model_df[
        "TimeStamp_dt"
    ]
    .dt.floor(
        f"{TIME_BLOCK_MINUTES}min"
    )
)

unique_blocks = sorted(
    model_df[
        "time_block"
    ]
    .dropna()
    .unique()
)

purged_rows = []
purged_predictions = {}

for feature_name, cols in FEATURE_SETS.items():

    X = model_df[
        cols
    ].astype(float)

    oof = np.full(
        len(model_df),
        np.nan,
    )

    usable_folds = 0

    for block in unique_blocks:

        block = pd.Timestamp(
            block
        )

        val_mask = (
            model_df[
                "time_block"
            ] == block
        )

        va_idx = np.where(
            val_mask.to_numpy()
        )[0]

        if len(
            va_idx
        ) == 0:
            continue

        purge_start = (
            block
            - pd.Timedelta(
                minutes=
                    PURGE_MINUTES
            )
        )

        purge_end = (
            block
            + pd.Timedelta(
                minutes=
                    TIME_BLOCK_MINUTES
                    + PURGE_MINUTES
            )
        )

        train_mask = ~(
            (
                model_df[
                    "TimeStamp_dt"
                ]
                >= purge_start
            )
            &
            (
                model_df[
                    "TimeStamp_dt"
                ]
                < purge_end
            )
        )

        val_groups = set(
            model_df.loc[
                val_mask,
                "sensor_group_id",
            ].tolist()
        )

        train_mask &= ~model_df[
            "sensor_group_id"
        ].isin(
            val_groups
        )

        tr_idx = np.where(
            train_mask.to_numpy()
        )[0]

        if (
            len(
                tr_idx
            ) == 0
            or y.iloc[
                tr_idx
            ].nunique()
            < 2
        ):
            continue

        model = make_model()

        model.fit(
            X.iloc[
                tr_idx
            ],
            y.iloc[
                tr_idx
            ],
        )

        pred = (
            model.predict_proba(
                X.iloc[
                    va_idx
                ]
            )[:, 1]
        )

        oof[
            va_idx
        ] = pred

        usable_folds += 1

    metrics = evaluate(
        y,
        oof,
    )

    purged_predictions[
        feature_name
    ] = oof

    purged_rows.append({
        "feature_set":
            feature_name,
        "n_features":
            len(cols),
        "usable_folds":
            usable_folds,
        **metrics,
    })

    print(
        f"{feature_name:28s} | "
        f"PR={metrics['PR_AUC']:.4f} | "
        f"R80 P={metrics['R80_precision']:.3f}, "
        f"R={metrics['R80_recall']:.3f} | "
        f"R90 P={metrics['R90_precision']:.3f}, "
        f"R={metrics['R90_recall']:.3f} | "
        f"Top5 P={metrics['Top5_precision']:.3f}, "
        f"R={metrics['Top5_recall']:.3f}"
    )

purged_df = pd.DataFrame(
    purged_rows
)

purged_df.to_csv(
    OUT_DIR
    / "purged_results.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 8. 통합 비교
# =============================================================================

banner(
    "7) Group + Purged 통합 비교"
)

combined = group_df.merge(
    purged_df,
    on=[
        "feature_set",
        "n_features",
    ],
    suffixes=(
        "_group",
        "_purged",
    ),
)

combined[
    "PR_min"
] = combined[
    [
        "PR_AUC_group",
        "PR_AUC_purged",
    ]
].min(
    axis=1
)

combined[
    "delta_PR_vs_base_group"
] = (
    combined[
        "PR_AUC_group"
    ]
    - float(
        combined.loc[
            combined[
                "feature_set"
            ] == "base_side",
            "PR_AUC_group",
        ].iloc[0]
    )
)

combined[
    "delta_PR_vs_base_purged"
] = (
    combined[
        "PR_AUC_purged"
    ]
    - float(
        combined.loc[
            combined[
                "feature_set"
            ] == "base_side",
            "PR_AUC_purged",
        ].iloc[0]
    )
)

combined = combined.sort_values(
    [
        "PR_min",
        "PR_AUC_group",
    ],
    ascending=False,
)

show_cols = [
    "feature_set",
    "n_features",
    "PR_AUC_group",
    "PR_AUC_purged",
    "PR_min",
    "delta_PR_vs_base_group",
    "delta_PR_vs_base_purged",
    "R80_precision_group",
    "R80_recall_group",
    "R80_precision_purged",
    "R80_recall_purged",
    "R90_precision_group",
    "R90_precision_purged",
    "Top5_precision_group",
    "Top5_recall_group",
]

print(
    combined[
        show_cols
    ]
    .round(4)
    .to_string(
        index=False
    )
)

combined.to_csv(
    OUT_DIR
    / "group_vs_purged.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 9. Temporal feature 결측률
# =============================================================================

banner(
    "8) Causal feature 품질"
)

quality_rows = []

for c in temporal_cols:

    quality_rows.append({
        "feature":
            c,
        "missing_rate":
            float(
                model_df[
                    c
                ]
                .isna()
                .mean()
            ),
        "mean":
            float(
                model_df[
                    c
                ]
                .mean(
                    skipna=True
                )
            ),
        "std":
            float(
                model_df[
                    c
                ]
                .std(
                    skipna=True
                )
            ),
    })

quality_df = (
    pd.DataFrame(
        quality_rows
    )
    .sort_values(
        "missing_rate"
    )
)

print(
    quality_df[
        "missing_rate"
    ]
    .describe()
    .round(4)
    .to_string()
)

quality_df.to_csv(
    OUT_DIR
    / "causal_feature_quality.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 10. 생성 데이터 저장
# =============================================================================

cycle.to_csv(
    OUT_DIR
    / "cn7_cycle_606_with_causal_features.csv",
    index=False,
    encoding="utf-8-sig",
)

model_df.to_csv(
    OUT_DIR
    / "cn7_1211_with_causal_features.csv",
    index=False,
    encoding="utf-8-sig",
)


# =============================================================================
# 11. 자동 요약
# =============================================================================

banner(
    "9) 자동 요약"
)

best_group = (
    group_df
    .sort_values(
        "PR_AUC",
        ascending=False,
    )
    .iloc[0]
)

best_purged = (
    purged_df
    .sort_values(
        "PR_AUC",
        ascending=False,
    )
    .iloc[0]
)

best_robust = (
    combined.iloc[0]
)

base_row = combined[
    combined[
        "feature_set"
    ] == "base_side"
].iloc[0]

lines = []

lines.append(
    f"- Cycle: {len(cycle)}개, "
    f"session: {cycle['session_id'].nunique()}개"
)

lines.append(
    f"- 생성 causal feature: "
    f"{len(temporal_cols)}개"
)

lines.append(
    f"- Baseline base_side: "
    f"Group PR={base_row['PR_AUC_group']:.4f}, "
    f"Purged PR={base_row['PR_AUC_purged']:.4f}"
)

lines.append(
    f"- Group 최고: "
    f"{best_group['feature_set']} | "
    f"PR={best_group['PR_AUC']:.4f}, "
    f"R80 Precision={best_group['R80_precision']:.4f}, "
    f"R80 Recall={best_group['R80_recall']:.4f}"
)

lines.append(
    f"- Purged 최고: "
    f"{best_purged['feature_set']} | "
    f"PR={best_purged['PR_AUC']:.4f}, "
    f"R80 Precision={best_purged['R80_precision']:.4f}, "
    f"R80 Recall={best_purged['R80_recall']:.4f}"
)

lines.append(
    f"- PR_min 최고: "
    f"{best_robust['feature_set']} | "
    f"Group={best_robust['PR_AUC_group']:.4f}, "
    f"Purged={best_robust['PR_AUC_purged']:.4f}"
)

if (
    best_purged[
        "PR_AUC"
    ]
    > base_row[
        "PR_AUC_purged"
    ]
):
    lines.append(
        "- 결론: causal 변화량 feature가 "
        "시간구간 일반화에 개선 신호를 보였다."
    )
else:
    lines.append(
        "- 결론: causal 변화량만으로는 "
        "시간구간 일반화 문제가 해결되지 않았다."
    )

lines.append(
    "- 다음 단계는 결과에 따라 "
    "temporal feature 축소/선별 또는 test 원본 복원으로 진행한다."
)

for line in lines:
    print(
        line
    )

with open(
    OUT_DIR
    / "causal_feature_summary.txt",
    "w",
    encoding="utf-8",
) as f:

    f.write(
        "\n".join(
            lines
        )
    )


# =============================================================================
# 완료
# =============================================================================

banner(
    "완료"
)

print(
    "결과 폴더:",
    OUT_DIR,
)

print()

print(
    "핵심 파일:"
)

print(
    "  1) group_vs_purged.csv"
)

print(
    "  2) group_results.csv"
)

print(
    "  3) purged_results.csv"
)

print(
    "  4) cn7_cycle_606_with_causal_features.csv"
)

print(
    "  5) cn7_1211_with_causal_features.csv"
)

print(
    "  6) causal_feature_quality.csv"
)

print(
    "  7) causal_feature_summary.txt"
)

print()

print(
    "실행 후 7) Group + Purged 통합 비교부터 "
    "9) 자동 요약까지만 보내주세요."
)



CN7 09 - Causal Process Change Features
rows=1211, defects=17, base_sensors=23
constant removed: ['Clamp_Open_Position']

1) Cycle 재구성
row count: 1211
cycle count: 606
session count: 2

            cycles               start                 end
session_id                                                
1              188 2020-10-16 04:57:47 2020-10-16 08:05:18
2              418 2020-10-19 23:24:46 2020-10-20 06:20:14

2) Causal feature 생성
temporal features: 138
예시: ['CF_Injection_Time_delta1', 'CF_Injection_Time_dev3', 'CF_Injection_Time_dev5', 'CF_Injection_Time_z5', 'CF_Injection_Time_dev10', 'CF_Injection_Time_z10', 'CF_Filling_Time_delta1', 'CF_Filling_Time_dev3', 'CF_Filling_Time_dev5', 'CF_Filling_Time_z5', 'CF_Filling_Time_dev10', 'CF_Filling_Time_z10', 'CF_Plasticizing_Time_delta1', 'CF_Plasticizing_Time_dev3', 'CF_Plasticizing_Time_dev5']

3) Row-level merge
model_df: (1211, 194)

4) Feature sets
base_side                   : 24 features
delta_side                  : 24 feat